In [ ]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("canonicalization_module.py")))
from quineformer import CanonicalizationModule
import torch.nn.functional as F

In [3]:

from datasets import load_dataset
import numpy as np

# ── Reference text dataset for MLM perplexity & KL-divergence training ────────
# wikitext-103-raw-v1 is drawn from Wikipedia "Good/Featured" articles —
# the same distribution as BERT's pretraining corpus. We only need a small
# slice (~65K tokens), so we load once and keep the tensor on CPU.

REF_SEQ_LEN   = 128   # tokens per chunk (identical to BERT pretraining)
REF_N_SEQS    = 512   # =65,536 tokens total; slice forward-pass mini-batches from this
MLM_MASK_PROB = 0.15  # standard BERT masking rate (for perplexity evaluation)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

wiki = load_dataset(
    "wikitext",
    "wikitext-103-raw-v1",
    split="train"
)

# Concatenate all non-empty texts, tokenise without special tokens, then chunk.
all_ids: list = []
for item in wiki:
    text = item["text"].strip()
    if not text:
        continue
    all_ids.extend(
        tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
        )["input_ids"]
    )
    if len(all_ids) >= REF_SEQ_LEN * REF_N_SEQS:
        break

# Truncate to an exact multiple of REF_SEQ_LEN and reshape to (N, SEQ_LEN).
n_tokens = (len(all_ids) // REF_SEQ_LEN) * REF_SEQ_LEN
ref_ids  = torch.tensor(all_ids[:n_tokens], dtype=torch.long).reshape(-1, REF_SEQ_LEN)
ref_ids  = ref_ids[:REF_N_SEQS]   # cap at REF_N_SEQS
print(f"Reference corpus: {ref_ids.shape[0]} sequences × {REF_SEQ_LEN} tokens  "
      f"({ref_ids.numel():,} tokens total)")

# ── Unmasked batch (for KL-divergence loss during training) ──────────────────
# At training time, sample a mini-batch of 4–8 sequences like:
#   idx   = torch.randint(len(ref_ids), (TRAIN_BATCH_SIZE,))
#   batch = ref_ids[idx].to(DEVICE)          # move to GPU on the fly

# ── Static masked batch (for MLM perplexity evaluation) ──────────────────────
rng = np.random.default_rng(seed=42)
_mask = torch.from_numpy(rng.random(ref_ids.shape) < MLM_MASK_PROB)  # (N, L) bool

ref_masked_ids = ref_ids.clone()
ref_masked_ids[_mask] = tokenizer.mask_token_id  # 103

ref_mlm_labels = ref_ids.clone()
ref_mlm_labels[~_mask] = -100  # CrossEntropyLoss ignores -100

n_masked = int(_mask.sum())
print(f"MLM eval batch : {n_masked:,} / {ref_ids.numel():,} tokens masked "
      f"({n_masked / ref_ids.numel() * 100:.1f} %)")
print(f"Sample (first 20 tokens): {tokenizer.decode(ref_ids[0, :20].tolist())!r}")


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Reference corpus: 512 sequences × 128 tokens  (65,536 tokens total)
MLM eval batch : 9,767 / 65,536 tokens masked (14.9 %)
Sample (first 20 tokens): '= valkyria chronicles iii = senjo no valkyria 3 : unrecorded chronicles'
